In [0]:
# ============================================================
#  RAW → SILVER: BODEGAS
# ============================================================

STORAGE_ACCOUNT  = "stdatacolake"
CONTAINER_RAW    = "raw-zone"
CONTAINER_SILVER = "silver-zone"

RAW_PATH    = f"abfss://{CONTAINER_RAW}@{STORAGE_ACCOUNT}.dfs.core.windows.net/ORACLE"
SILVER_PATH = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bodegas"

In [0]:
# ── LEER CSV DESDE RAW ────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema_bodegas = StructType([
    StructField("id_bodega",      IntegerType(), nullable=False),
    StructField("nombre_bodega",  StringType(),  nullable=True),
    StructField("departamento",   StringType(),  nullable=True),
    StructField("ciudad",         StringType(),  nullable=True),
    StructField("capacidad_max",  IntegerType(), nullable=True),
])

df_raw = spark.read \
    .schema(schema_bodegas) \
    .option("header", "true") \
    .csv(f"{RAW_PATH}/Datos_bodega.csv")

print(f"✅ Raw leído: {df_raw.count()} filas")
df_raw.show(5)

In [0]:
# ── LIMPIEZA Y TRANSFORMACIONES ───────────────────────────────
from pyspark.sql.functions import col, trim, initcap, current_timestamp

df_silver = df_raw \
    .filter(col("id_bodega").isNotNull()) \
    .filter(col("capacidad_max") > 0) \
    .withColumn("nombre_bodega",  trim(col("nombre_bodega"))) \
    .withColumn("departamento",   initcap(trim(col("departamento")))) \
    .withColumn("ciudad",         initcap(trim(col("ciudad")))) \
    .dropDuplicates(["id_bodega"]) \
    .withColumn("_updated_at", current_timestamp())

print(f"✅ Silver listo: {df_silver.count()} filas")
df_silver.show(5)

In [0]:
# ── GUARDAR COMO DELTA EN SILVER ─────────────────────────────
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(SILVER_PATH)

print(f"✅ Guardado en Silver: {SILVER_PATH}")